Nel retrieval moderno, il principale collo di bottiglia nel processare una query è dato dal **calcolo degli score**. Una volta recuperate le posting list dei termini della query infatti il sistema deve calcolare quanto ogni documento (posting) sia rilevante rispetto alla query secondo un dato modello (es. VSM, BM25, LM...). **Questo calcolo può essere molto costoso perché le posting list che recuperiamo sono l'OR dei termini della query, quindi potenzialmente potremmo dover prendere in considerazione moltissimi documenti**

es. q = "information retrieval models" --> dovremo prendere in considerazione anche i documenti che contengono solo "information" o solo "retrieval" o solo "models", oltre a quelli che contengono più termini della query.

Tuttavia ovviamente nei motori di ricerca esiste un vincolo molto stringente di latenza da rispettare: la risposta deve arrivare in pochi millisecondi --> il sistema non può permettersi di calcolare lo score per ogni documento candidato --> è necessario un meccanismo di **pruning** che ci permetta di **evitare di calcolare lo score dei documenti che non rientreranno nella top-K** (dove $K$ è il numero di documenti che vogliamo restituire come risultato della query all'utente).

Il problema che si affronta in questa sezione è quindi: come ridurre il numero di documenti da valutare, senza però peggiorare troppo la qualità dei risultati?

Distinguiamo in questo senso **due tipi di pruning/ranking**:
- **Safe Ranking**: un metodo è detto safe se **garantisce che i $K$ documenti restituiti siano esattamente gli stessi che avremmo ottenuto senza pruning**. 
- **Non-Safe Ranking**: si parla invece di metodo non-safe se non dà questa garanzia: può avvenire che per velocizzare il calcolo venga eliminato un documento che in realtà sarebbe rientrato nella top-K. In cambio però ci si aspetta che metodi di questo tipo siano molto veloci

**Ma è accettabile usare metodi non-safe?** Spesso in pratica sì, soprattutto perché nei sistemi reali le funzioni di ranking non rappresentano mai una verità assoluta, ma sono solo un'approssimazione di quanto un documento sia rilevante per una query.

In questo senso si ricorda infatti che **la ranking function è solo una proxy**: anche se calcolassimo perfettamente lo score secondo la funzione, questo potrebbe non riflettere davvero il bisogno informativo dell'utente per una miriade di motivi (es. query ambigua, il fatto che la funzione di ranking è un'approssimazione etc...). Quindi in definitiva **perdere la top-K esatta non è necessariamente grave, se i documenti restituiti sono comunque molto buoni per l'utente**.

## Efficient Ranking: Ordinamento vs Selezione
Immaginiamo di trovarci nel VSM (il discorso è analogo per gli altri modelli, cambia lo score e non c'è rappresentazione vettoriale). 

Con VSM il retrieval viene ricondotto a un problema geometrico: ogni documento è rappresentato come vettore in un iperspazio di cardinalità pari al numero di termini del vocabolario, e il valore associato è il peso tf-idf. Allo stesso modo la query è rappresentata come vettore, e il ranking si basa sulla vicinanza tra vettore query e vettore documento sfruttando una certa misura di similarità (noi abbiamo visto la cosine similarity, per cui query e documento sono simili se puntano più o meno nella stessa direzione).

Sia $J$ il numero di documenti candidati (quelli che contengono almeno un termine della query, e che quindi hanno cosine score diverso da zero) e $K$ il numero di documenti che vogliamo restituire. Allora il problema dell'efficient ranking ha due parti distinte:
1. **Selezione**: identificare e calcolare efficacemente lo score di un sottinsieme dei $J$ documenti candidati. Si vorrebbe in particolare, dato che $J$ potrebbe essere come detto prima molto grande, identificare un sottinsieme di documenti $A$ che sia tale che $K \leq |A| \ll J$ e che contenga comunque i documenti più rilevanti (o almeno quelli che si avvicinano di più alla top-K).
2. **Ordinamento**: scegliere efficacemente i $K$ documenti più rilevanti tra quelli di $A$

### Ordinamento
Immaginiamo di avere già effettuato pruning e di avere un insieme $A$ di documenti candidati, dove $|A| = m \gt K$. A questo punto dobbiamo trovare i $K$ documenti più rilevanti tra quelli di $A$. L'idea naive sarebbe ordinare tutti i documenti di $A$ in base al loro score e restituire i primi $K$. Tuttavia questo ha complessità $O(m \log m)$, che può essere troppo elevata se $m$ è grande.

L'idea per risolvere il problema efficacemente è quindi utilizzare un max-heap: si ricorda che un max-heap è una struttura dati ad albero binario in cui ogni nodo ha valore maggiore o uguale rispetto a quello dei figli. 

Se manteniamo i documenti di $A$ in un max-heap ordinato per score, allora possiamo estrarre i $K$ documenti più rilevanti in tempo $O(K \log m)$ (ad ogni passo estraggo il massimo e riorganizzo l'heap di conseguenza in tempo logaritmico, per un totale di $K$ estrazioni). Costruire l'heap ha complessità $O(m)$, quindi in totale si ha una complessità di $O(m + K \log m)$.

Il vero bottleneck però non è l'ordinamento, ma il calcolo dello score. In questo senso approfondiamo alcune delle tecniche per la selezione dei documenti da valutare (ovvero il pruning). 

### (non-safe) Pruning
L'obiettivo è quindi **ridurre il numero di documento su cui calcolare lo score**. Come anticipato si vuole arrivare a costruire un insieme $A$ di documenti candidati tale che $K \leq |A| \ll J$, possibilmente in modo che in $A$ siano comunque presenti i documenti più rilevanti (o almeno quelli che si avvicinano di più alla top-K originale). Vediamo ora in particolare quelle tecniche che nel generare $A$ non garantiscono che la top-K finale sia esattamente la stessa che avremmo ottenuto senza pruning (**non-safe pruning**).

Il metodo base di pruning, che in realtà abbiamo già dato per scontato, è chiamato **Index Elimination**: grazie all'inverted index non consideriamo tutti i documenti della collezione per calcolare lo score, ma solo l'insieme $J$ dei documenti che contengono almeno un termine della query (OR tra le posting list dei termini della query). Questo è un primo filtro banale e safe in quanto i documenti che non contengono alcun termine della query, per modelli sintattici come quelli visti finora, avranno score zero.

Index Elimination però è il punto di partenza per due tecniche più selettive: 
1. **High-idf query terms only**: una query come "catcher in the rye" contiene parole che non sono informative allo stesso modo; mentre catcher e rye sono termini specifici, in e the sono stop word che appaiono in moltissimi documenti e che tendono a non identificare bene documenti rilevanti. In particolare termini di questo tipo hanno basso idf --> possiamo fare pruning prendendo solo i documenti con termini aventi idf sopra una certa soglia. Questo è un metodo non-safe, perché potremmo perdere documenti che contengono solo termini a basso idf ma che sono comunque rilevanti per la query e apparirebbero nella top-K originale.
2. **Docs containing many query terms**: qui l'intuizione sta nel fatto che se la query contiene molti termini, allora spesso i documenti migliori sono quelli che ne contengono diversi, non soltanto uno! Allora possiamo fare pruning imponendo una condizione del tipo "considero solo i documenti che contengono almeno $n$ termini della query". Si parla in questo senso di **soft conjuction**: non è un AND rigido, ma nemmeno un OR puro. Anche questo è un metodo non-safe, perché potremmo perdere documenti che contengono solo pochi termini della query ma che sono comunque rilevanti e apparirebbero nella top-K originale.

Prima di passare alle prossime tecniche di pruning, è importante dire che queste tecniche possono essere combinate e non esiste un metodo migliore in assoluto: la scelta dipende molto dal contesto (collezione, modello di ranking, tipo di query etc...) --> **benchmark** fondamentale. In particolare in questo caso la misura che sicuramente ha più senso massimizzare è la **recall**: infatti nel pruning il problema principale è la possibilità di lasciare fuori documenti rilevanti, e poiché la recall misura proprio la frazione di documenti rilevanti rispetto a tutti i documenti rilevanti, è la misura più adatta per valutare l'efficacia di un metodo di pruning.

#### Champion lists
Questa tecnica si basa sulla seguente idea: **per ogni termine del dizionario, salvare in anticipo i migliori $r$ documenti rispetto a quel termine** (dove con migliore si intende quelli con lo score più alto, nel caso VSM TF-IDF). In questo modo al momento del retrieval, per ogni termine $t$ della query invece di scorrere la posting list completa si usa solo la champion list.

Questa tecnica non è safe perché ogni champion list contiene solo i migliori $r$ documenti rispetto a quel singolo termine --> non stiamo prendendo in considerazione gli altri termini della query. Un documento potrebbe infatti essere molto buono nel calcolo dello score complessivo considerando tutti i termini della query, ma non essere tra i migliori $r$ per nessun singolo termine --> escluso dalle champion list e quindi non finisce nella top-K finale.

Un altro problema di questa tecnica è che $r$ deve essere scelto a priori, e se poi $K \gt r$ allora potremmo non avere abbastanza documenti per restituire la top-K. 

Altro potenziale problema ancora: usare champion list può essere pericoloso per alcuni documenti. Se un documento non compare in nessuna champion list per nessun termine, se il motore di ricerca usa solo champion list per rispondere alle query allora quel documento non potrà mai essere restituito come risultato --> diventa invisibile a qualsiasi ricerca, ma è comiunque indicizzato, finisce nel deep web ed è raggiungibile solo tramite link (anche se deep web in senso tecnico indica pagine che non sono proprio indicizzate dal motore, in questo caso il documento è presente nell'inverted index ma non è raggiungibile tramite query).

Come implementare le champion list a partire dall'inverted index? Molto semplice, per ogni termine abbiamo la posting list, e ogni posting contiene almeno docID e peso del termine nel documento (nel caso VSM TF-IDF). Basta ordinare la posting list in base al peso e prendere i primi $r$ documenti per costruire la champion list.

Per il funzionamento poi si hanno diverse possibilità: potremmo come detto scorrere solo le champion list dimenticandoci delle posting list originali, oppure per evitare il problema di $r \lt K$ potremmo concatenare le champion list con le posting list originali e scorrere prima le champion list, per poi continuare con le posting list originali se non abbiamo ancora trovato $K$ documenti (idea simile a quella del tiered index, di cui parleremo dopo).

#### Query independent document scores (Static Quality Scores)
Si introduce un concetto che sarà utile per capire le prossime tecniche di pruning: non tutti i documenti hanno lo stesso valore a priori della query: **due documenti possono avere lo stesso score rispetto a una query, ma uno dei due può essere più autorevole dell'altro.**

Ad esempio se cerco "climate change causes" potrei trovare un articolo scientifico molto citato, ma anche un post anonimo di bassa qualità. Magari entrambi contengono i termini della query, ma il primo è probabilmente più affidabile. 

Quindi il ranking dovrebbe tenere conto di due cose: 
$$ \text{rilevanza rispetto alla query} + \text{autorevolezza del documento} $$
Mentre la rilevanza in VSM è modellata dalla cosine similarity query/documento, l'autorevolezza è una proprietà come detto query-independent che può essere calcolata a priori in base a caratteristiche del documento del tipo:
- Se è pagina Wikipedia
- Se si tratta di un articolo di giornali particolarmente autorevoli
- Se si tratta di un paper molto citato
- PageRank: misura l'autorevolezza di una pagina web in base al numero e alla qualità dei link che la puntano 

A ogni documento $d$ viene quindi assegnato a priori un punteggio di autorevolezza $g(d) \in [0,1]$ (dove 1 è il massimo di autorevolezza). 

Un esempio di formula per calcolare g(d) in base alle citazioni del documento $d$ può essere banalmente $g(d) = \frac{\text{citations}(d)}{\text{max\_cit}}$ dove $\text{max\_cit}$ è il numero di citazioni del documento più citato della collezione. Questa formula è un po' grezza perché le citazioni sono per loro natura sbilanciate: pochi documenti hanno tantissime citazioni mentre molti ne hanno poche --> possiamo usare la formula con il log $g(d) = \frac{\log(\text{citations}(d)+1)}{\log(\text{max\_cit}+1)}$ per ridurre questo sbilanciamento.

Una volta capito come calcolare $g(d)$, possiamo usarlo per definire un nuovo score che tenga conto sia della rilevanza rispetto alla query che dell'autorevolezza del documento, questo è noto come **net-score**:
$$ \text{net-score}(d) = \alpha \cdot \text{cosine}(q,d) + (1 - \alpha) \cdot g(d) $$
dove $\alpha \in [0,1]$ è un parametro che bilancia l'importanza tra rilevanza e autorevolezza.

Ora il problema quindi diventa come trovare i $K$ documenti con net-score più alto in modo efficiente --> pruning basato su net-score.

Un'**idea molto importante** che può permettere di velocizzare il sistema è quella di **ordinare tutte le posting lists secondo $g(d)$ invece che in base al docID**. Normalmente infatti sappiamo che le posting lists sono ordinate in base al docID per facilitare intersezioni e altre amenità, ma questo serve soprattutto per modelli booleani e non è strettamente necessario per modelli basati su score come VSM. **Svantaggio: comunque non avere le posting ordinate per docID rende meno efficienti alcune operazioni come il calcolo dello score per tutti i termini della query** (infatti non potendo più scorrere le posting per docID non possiamo più calcolare lo score di un documento in un unico passaggio, ma dobbiamo scorrere tutte le posting list per cercare il docID corrispondente per ogni termine della query e sommare i contributi --> più passaggi e quindi più tempo).

L'ordinamento baato su $g(d)$ garantisce che tutte le posting list vengano scorse a partire dai documenti più autorevoli, **in questo modo i documenti con net-score più alto probabilmente compariranno presto!** (ricordiamo infatti che $\text{net-score} = \alpha \cdot \text{cosine}(q,d) + (1 - \alpha) \cdot g(d)$, quindi un documento con alto $g(d) parte avvantaggiato).

Questa logica può essere utilizzata **in applicazioni time-bound**: immaginiamo che il sistema abbia un vincolo di latenza 50 ms (deve rispondere peffò entro 50 ms) --> il sistema può iniziare a scorrere le posting list ordinate per $g(d)$ e calcolare il net score dei documenti, e restituire dopo 50 ms i migliori risultati trovati fino a quel momento. Dal momento che i le posting come detto sono ordinate per $g(d)$, è più probabile che i risultati trovati all'inizio siano buoni. Ovviamente questa tecnica non è safe (se mi fermo troppo presto potrei non aver trovato i documenti migliori, magari non autorevoli ma comunque con score sintattico molto più alto rispetto alla query).

**Questa idea può essere anche combinata con le champion list**: invece di costruire champion list basate sullo score sintattico, le possiamo costruire in base al net-score --> anche se un documento ha un tf-idf (nel caso VSM) basso, se è molto autorevole potrebbe comunque finire nella champion list. Anche in questo caso non-safe per gli stessi motivi della champion list normale.

#### Cluster pruning
L'idea molto in generale è la seguente: invece di lavorare solo sulle posting list dei termini, l'idea è di organizzare anzitutto la collezione in gruppi di documenti simili. Poi, quando arriva la query, si cerca solo nei gruppi che sono più simili alla query.

Immaginiamo che tutti i documenti siano punti in uno spazio vettoriale, come in VSM (nota bene: generalizzabile anche con gli altri modelli, basta che ogni documento abbia uno score e si definisca una misura di similarità) --> documenti simili sono vicini nello spazio. **Cluster pruning** funziona come segue:
- dalla collezione di documenti si scelgono $l$ (ad esempio $l=\sqrt{N}$, dove $N$ è il numero totale di documenti) documenti particolarmente rappresentativi, detti **Leader**
- ogni altro documento viene assegnato al leader più vicino, formando così $l$ cluster di documenti (K-means è un algoritmo classico per fare questo tipo di clustering)
- al momento del retrieval, si calcola la similarità tra la query e i leader, e si seleziona il leader più simile --> si calcolano gli score solo del leader in questione e dei documenti suoi followers, e si restituiscono i migliori $K$ tra questi

<img src="img/cluster_pruning.png" alt="cluster pruning" width="400">

In questo modo si riduce drasticamente il numero di documenti su cui calcolare lo score, ma ovviamente non è safe: potremmo perdere documenti rilevanti che appartengono a cluster i cui leader non sono tra i $n$ più simili alla query.

Ma come scegliere i leader? Un'idea buona è quella di fare random sampling, scegliendo un campione casuale di $l$ documenti che rappresentino bene la distribuzione originale della collezione. Fare random sampling è l'ideale perché è veloce e permette di ottenere documenti rappresentativi.

Di Cluster Pruning esistono diverse varianti, ad esempio:
1. Ogni follower potrebbe essere associato a più ($b_1$) leader, in modo da aumentare le possibilità di recuperare documenti rilevanti (ma ovviamente aumenta anche il costo computazionale)
2. Invece di selezionare un solo leader, potremmo prendere i $b_2$ più simili e ripetere il ragionamento calcolando quindi lo score per ognuno degli $l$ cluster
3. Si può costruire una struttura gerarchica di cluster, fatta ad esempio di leader; sub-leader e followers. in questo modo possiamo ridurre il numero di leader da considerare al momento di retrieval ed evitare di dover fare $l$ confronti tra query e leader (es. ho un milione di documenti, invece di prendere $\sqrt{10^6} = 1000$ leader e direttamente followers, costruisco una gerarchia con 100 leader e 10 sub-leader per ogni leader, in questo modo al momento del retrieval faccio solo 100 confronti invece di 1000)

Nota che $b_1$ e $b_2$ sono parametri da scegliere a priori diversi in quanto $b_1$ riguarda la fase di costruzione dei cluster, mentre $b_2$ riguarda la fase di retrieval. 

#### Tiered Index
Un **tiered index** è un inverted index diviso in più livelli (tiers).Per ogni termine invece di avere una singola posting list completa, **si hanno più posting list ordinate per importanza**. 

**L'idea è: prima si cerca nei documenti più promettenti (presenti nei primi tier), se non se ne trovano abbastanza, allora si passa ai tier inferiori**.

La versione più semplice è quella che mantiene solo due livelli: per ogni termine si mantengono solo due posting list: high se contiene documenti più importanti per il termine, low se contiene i restanti. **Cosa si intende per documenti più importanti?** Si torna al concetto di champion lists con net-score: i documenti più importanti per un termine sono quelli con net-score più alto di una certa threshold rispetto a quel termine --> una high list è simile a una champion list (contiene i documenti migliori per quel termine). 

Query time i tiered index funzionano così:
1. per ogni termine si guardano solo le high list e si calcolano gli score dei documenti che vi appaiono
2. se trovo più di $K$ candidati, seleziono i top-K e li restituisco
3. se invece non trovo abbastanza candidati, allora passo a guardare anche le low list, e così via fino a quando non ho trovato $K$ documenti da restituire

Diverso dalle champion list perché lì si guardavano SOLO i migliori $r$, mentre con le tiered si mantengono anche gli altri documenti, ma in livelli più bassi.

Il criterio per decidere l'importanza a priori di un documento può essere solo la qualità statica $g(d)$ oppure come anticipato il net-score basato su $g(d)$ e sulla rilevanza sintattica rispetto a quel singolo termine.

<img src="img/tiered_index.png" alt="tiered index" width="400">

L'uso dei tiered index velocizza il retrieval perché spesso i risultati migliori si trovano nei primi tier, e quindi invece di attraversare tutte la posting list ci si ferma tipicamente prima. Ovviamente però non è safe, perché potremmo perdere documenti rilevanti che si trovano nei tier più bassi (il calcolo del net-score in questo caso è solo a priori rispetto al singolo termine, non possiamo sapere se poi quando arriva la query rispetto agli altri termini un documento low tier ha un net-score effettivo molto alto).

#### Impact-ordered postings
Seguendo un'idea praticamente analoga all'ordinamento delle postings rispetto a $g(d)$, **si potrebbe pensare di ordinare le posting list in base al peso sintattico del termine in quel documento** (es. TF-IDF nel caso VSM). Chiamiamo questo peso $wf_{t,d}$. L'idea è che se un termine contribuisce molto allo score sintattico a priori di un documento --> è più probabile che quel documento sia rilevante per una query che contiene quel termine --> ha senso analizzarlo tra i primi. 

Si potrebbe quindi anche pensare di ordinare le posting in base al net-score "aprioristico" (sia chiara la differenza tra net-score normale e "aprioristico": normale quando guardiamo alla query e quindi mischiamo score effettivo e autorevolezza, aprioristico quando guardiamo solo al singolo termine e quindi mischiamo lo score **legato al singolo termine rispetto al documento** e autorevolezza).

**Svantaggio**: lo stesso visto prima, si perde l'ordinamento delle postings in base al DocID e quindi si perde la possibilità di fare merge/intersezioni efficienti per il calcolo dello score complessivo rispetto alla query. 

Come mitigare questa inefficienza? Due idee:
1. **Early Termination**: quando si attraversano le posting list di un termine ordinate come detto sopra, potremmo:
   1. fermarci dopo un numero fissato $r$ di postings, per poi prendere l'unione dei documenti così trovati per poi calcolarci lo score complessivo (in questo modo evito l'inefficienza di dover scorrere di volta in volta tutte le posting list per capire se un docID è presente o meno per calcolarne lo score complessivo)
   2. fermarsi non appena il peso del termine di un documento scende sotto una certa soglia e prenderre l'unione dei documenti così trovati per poi calcolarci lo score complessivo (stessa motivazione del punto precedente)
2. **idf-ordered terms**: questa seconda idea riguarda invece l'ordine in cui processo i termini della query. Si processano prima i termini con idf più alto e si aggiorna lo score dei documenti di volta in volta. Man mano che si processano termini con idf più basso, **quando si osserva che lo score di un documento non cambia entro una certa soglia**, allora ci si ferma e si restituiscono i documenti con score più alto fino a quel punto. L'idea è che sono i termini con idf più alto quelli che contribuiscono maggiormente allo score, quindi se dopo aver processato questi termini vedo che lo score di un documento non è cambiato molto, allora è improbabile che i termini con idf più basso possano farlo diventare rilevante e mi fermo. 


### (safe) Pruning
Finora abbiamo visto tecniche non-safe, che sono sicuramente veloci ma non garantiscono che un documento che sarebbe rientrato nella top-K originale non venga escluso. Con safe ranking invece vogliamo garantire che i $K$ documenti restituiti siano esattamente gli stessi che avremmo ottenuto senza pruning.

#### WAND (Weighted AND o bacchetta magica)
WAND è un algoritmo di pruning safe **DAAT** (Document-At-A-Time), ossia ragiona documento per documento invece che termine per termine:
- Term At a time (TAAT): prima si elabora tutta la posting list di un termine, aggiornando gli score dei documenti, per poi passare a quella del termine successivo
- Document At a time (DAAT): si guarda un possibile documento candidato e **si decide se calcolarne lo score completo oppure scartarlo senza calcolarlo**

Durante la ricerca, WAND mantiene un **threshold corrente** che rappresenta lo score del $K$-esimo miglior documento trovato fino a quel momento per la query corrente. Questo valore come vedremo aumenta ad ogni iterazione dell'algoritmo per cui la lista top-K è stata aggiornata. All'inizio dell'algoritmo, il threshold è inizializzato a zero.

es. supponiamo $K=3$ e abbiamo trovato con la scorsa query i documenti $d_{10}$ con score 9.9, $d_{31}$ con score 7.2 e $d_{25}$ con score 6.8 --> threshold = 6.8. Intuitivamente quindi WAND ci dice che un nuovo documento, per entrare nella top-3, deve fare meglio di 6.8. Se un documento al massimo può ottenere 5.4, allora non ha senso calcolarne lo score completo!

Sta qui la **natura safe di WAND**: un documento viene scartato solo quando si può dimostrare che lo score massimo possibile ottenibile dal documento è minore di un certo threshold --> se un documento viene scartato, è perché è impossibile che possa entrare nella top-K, quindi non stiamo perdendo documenti rilevanti.

**WAND funziona su inverted index con postings ordinate per docID crescente**. Ogni posting list ha un puntatore corrente a una posting, chiamato **finger**, che ci indica dove siamo arrivati nella lista e che si muove solo in avanti. Si assume l'invariante per cui tutto ciò che sta a sinistra dei finger è già stato deciso: un documento a sinistra può essere stato o scartato tramite pruning oppure valutato con score completo. 

<img src="img/wand.png" alt="wand" width="200">

(nell'esempio i numeri rappresentano i docID dei documenti che contengono quel termine, a cui ogni finger punta in questo dato momento.)

Si assume inoltre che l'inverted index sia implementato in modo che sia presente un **iteratore del tipo "go to docID $\ge$ X"**, ossia che ci permetta di muoverci nella posting list liberamente da un docID a un altro a patto che sia maggiore o uguale di un certo valore **X**.

es. se sono nella posting list di catcher e volessi saltare direttamente almeno al documento con docID 589, devo essere in grado di farlo senza dover scorrere tutta la posting list fino a quel punto.

Per ogni termine $t$ della query, WAND mantiene un **upper bound** $UB_t$, che rappresenta **il massimo contributo che il termine $t$ può ancora dare a qualunque documento non ancora processato** (dove con contributo si intende il peso del termine a priori in quel documento, es. TF-IDF nel caso VSM). 

Nell'esempio qui sotto il finger si trova in corrispondenza del documento con docID 29 --> tutti i documenti prima sono stati processati, il suo $UB_t$ è aggiornato come il massimo contributo che il termine $t$ può dare tra i documenti ancora a destra del finger. Chiaramente $UB_t$ allo scorrere del finger **è monotona decrescente: diminuisce solo quando il documento che dava contributo massimo viene superato**.

<img src="img/ub.png" alt="upper bound" width="400">

**Perché gli upper bound permettono il pruning**? Supponiamo che la query abbia i quattro termini "catcher in the rye" e per ogni termine si abbia upper bounds $UB_{catcher} = 2.3$, $UB_{rye} = 1.8$, $UB_{in} = 3.3$ e $UB_{the} = 4.3$. Se un documento contiene **tutti questi termini**, allora il suo **massimo score TEORICO possibile** sarebbe $2.3 + 1.8 + 3.3 + 4.3 = 11.7$. Ma se un documento può contenere solo alcuni dei termini prima di un certo "pivot" (lo vediamo meglio nell'esempio tra poco), allora sommo solo gli upper bound relativi a quei termini. **L'idea è che se anche sommando tutti gli upper bound di quei termini non arrivo alla soglia, allora nessun documento nella zona prima del pivot può entrare nella top-K.**

Vediamo l'applicazione di WAND per capire davvero sto funzionamento. 

Prendiamo sempre in considerazione la query "catcher in the rye" con upper bound $UB_{catcher} = 2.3$, $UB_{rye} = 1.8$, $UB_{in} = 3.3$ e $UB_{the} = 4.3$. I finger sono posizionati come segue: 

<img src="img/wand_example.png" alt="wand example" width="400">

La soglia corrente è 6.8. **Per prima cosa WAND ordina i termini in base alla posizione dei finger**. In questo caso l'ordine è "catcher" (docID 273), "rye" (docID 304), "in" (docID 589), "the" (docID 762). **Dopodiché inizia a sommare gli upper bound da sinistra verso destra finché non trova una somma che supera il threshold**:
- "catcher": 2.3
- "catcher" + "rye": 2.3 + 1.8 = 4.1
- "catcher" + "rye" + "in": 2.3 + 1.8 + 3.3 = 7.4

Il termine per cui la somma supera il threshold (nel nostro caso "in") viene definito **pivot**. Il pivot in questo caso ha docID pari a 589. **Il concetto chiave a questo punto è che un docID minore di 589 non può comparire dentro "in", perché il finger di "in" è già a 589 e tutto ciò che sta prima è già stato processato**, e tantomeno non può comparire in "the" perché il finger di "the" è già a 762.

Ma quindi un documento tra 273 e 588 può ricevere contributi al massimo da "catcher" + "rye" (perché al più contiene solo questi due termini), e la loro somma è 4.1, che è minore del threshold --> **tutti i documenti con docID tra 273 e 588 sono hopeless, non possono entrare nella top-K e quindi possono essere scartati senza calcolarne lo score completo**.

Quindi **possiamo spostare i finger di catcher e rye direttamente a 589 o oltre** (per questo ci serviva l'indicizzazione veloce!). A questo punto **WAND si chiede se valga la pena calcolare lo score completo del documento con docID 589. Se la somma degli upper bound dei termini che contengono quel documento supera il threshold, allora sì, calcola lo score** (che si ricorda non essere la somma degli upper bound, ma la somma dei contributi reali dei termini alla query in base al modello di scoring). **Poi se lo score entra nella top-K, aggiorno il threshold di conseguenza e ripeto il processo, altrimenti se lo score non superava il threshold/non entrava nella top K ripeto il processo senza aggiornare il threshold.**

<p align="center">
    <img src="img/we2.png" alt="wand example 1" width="45%">
    <img src="img/we3.png" alt="wand example 2" width="45%">
</p>

WAND non richiede di calcolare tutti gli score dei documenti perché chiaramente moltissimi vengono scartati prima. Inoltre, esattamente come gli altri metodi, WAND non funziona solo con la cosine similarity: **funziona per qualsiasi scoring function che può essere scritta come somma dei contributi per termine** (come ad esempio BM25 ed LM), lo scoring additivo è necessario perché WAND si basa sull'idea di upper bound per ogni termine e la loro somma per decidere se un documento è hopeless o meno.

Inoltre **WAND funziona meglio per query lunghe**, infatti in una query lunga molti documenti contengono solo alcuni termini, e se quei termini non bastano a superare il threshold li posso scartare. Con query corte, come quelle con un solo termine, c'è molto meno margine (con un solo termine guardo di volta in volta solo se il suo upper buond supera il threshold, si perde il senso dell'algoritmo).

WAND come detto è safe, ma esistono varianti che magari combinano idee viste prima per favorire la velocità che lo rendono non-safe. Ad esempio si potrebbe pensare di usare un threshold più alto rispetto a quello che avremmo con WAND normale, in modo da scartare più documenti --> però in questo modo potremmo perdere documenti rilevanti che avrebbero superato il threshold originale ma non quello più alto.